# 05 — Classic ML Bonus: Crop Prediction from Sensor Data

Companion notebook to `06-classic-ml-bonus-crop-prediction-case-study.md`.

Builds a small **synthetic** tabular dataset (rainfall, temperature, soil moisture, humidity, pH ->
recommended crop) with real-ish agronomic structure baked in via generation rules, then trains and
evaluates a `RandomForestClassifier`, including a feature-importance breakdown.

Fully offline, CPU only, runs in a few seconds.


## 1. Synthesize a sensor-reading dataset with agronomic structure

Real crop-recommendation data would come from historical sensor readings + agronomist labels. Here
we generate synthetic readings per crop from a distribution with a crop-specific mean/spread, so the
classes are learnable but not trivially separable (some noise/overlap between similar crops).


In [1]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

# Each crop's typical growing conditions: (rainfall_mm, temperature_c, soil_moisture_pct, humidity_pct, ph)
# Values are illustrative agronomic ballparks, not sourced from a real dataset.
CROP_PROFILES = {
    "rice":   {"rainfall": (200, 40), "temperature": (27, 3), "soil_moisture": (80, 8), "humidity": (85, 6), "ph": (6.0, 0.5)},
    "wheat":  {"rainfall": (75,  20), "temperature": (18, 4), "soil_moisture": (45, 8), "humidity": (55, 8), "ph": (6.5, 0.4)},
    "maize":  {"rainfall": (110, 25), "temperature": (24, 3), "soil_moisture": (55, 8), "humidity": (60, 8), "ph": (6.2, 0.4)},
    "cotton": {"rainfall": (90,  25), "temperature": (28, 3), "soil_moisture": (40, 8), "humidity": (50, 8), "ph": (7.0, 0.5)},
    "millet": {"rainfall": (55,  20), "temperature": (30, 3), "soil_moisture": (30, 8), "humidity": (45, 8), "ph": (6.8, 0.5)},
}
FEATURES = ["rainfall", "temperature", "soil_moisture", "humidity", "ph"]

rows = []
N_PER_CROP = 120
for crop, profile in CROP_PROFILES.items():
    for _ in range(N_PER_CROP):
        row = {feat: rng.normal(mean, std) for feat, (mean, std) in profile.items()}
        row["crop"] = crop
        rows.append(row)

df = pd.DataFrame(rows)
# Clip to physically plausible ranges
df["rainfall"] = df["rainfall"].clip(lower=0)
df["soil_moisture"] = df["soil_moisture"].clip(0, 100)
df["humidity"] = df["humidity"].clip(0, 100)
df["ph"] = df["ph"].clip(0, 14)

print(f"Dataset shape: {df.shape}")
df.sample(5, random_state=1)


Dataset shape: (600, 6)


,rainfall,temperature,soil_moisture,humidity,ph,crop
446,102.127044,24.767316,29.704740,62.662636,6.538260,cotton
404,90.367003,25.630041,41.039439,49.441610,6.333263,cotton
509,45.257729,29.064326,46.165118,31.120191,7.049894,millet
455,13.405503,24.165735,55.089513,45.627698,6.007928,cotton
201,81.804200,20.680317,42.001268,61.049985,6.651537,wheat


## 2. Train/test split (stratified) and Random Forest training

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

X = df[FEATURES]
y = df["crop"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

model = RandomForestClassifier(n_estimators=200, max_depth=None, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}\n")
print(classification_report(y_test, y_pred))


Accuracy: 0.793

              precision    recall  f1-score   support

      cotton       0.72      0.60      0.65        30
       maize       0.80      0.67      0.73        30
      millet       0.84      0.87      0.85        30
        rice       0.97      1.00      0.98        30
       wheat       0.66      0.83      0.74        30

    accuracy                           0.79       150
   macro avg       0.80      0.79      0.79       150
weighted avg       0.80      0.79      0.79       150



## 3. Confusion matrix — which crops get confused with which?

For a recommendation-style output, *which* classes get confused matters as much as the aggregate
accuracy number (see chapter 06's discussion of why).


In [3]:
from sklearn.metrics import confusion_matrix

labels_sorted = sorted(CROP_PROFILES.keys())
cm = confusion_matrix(y_test, y_pred, labels=labels_sorted)
cm_df = pd.DataFrame(cm, index=labels_sorted, columns=labels_sorted)
cm_df.index.name = "actual"
cm_df.columns.name = "predicted"
cm_df


predicted,cotton,maize,millet,rice,wheat
actual,,,,,
cotton,18,3,5,0,4
maize,0,20,0,1,9
millet,4,0,26,0,0
rice,0,0,0,30,0
wheat,3,2,0,0,25


## 4. Feature importance

In [4]:
importances = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False)
print("Feature importances (higher = more influential in the model's splits):\n")
print(importances)


Feature importances (higher = more influential in the model's splits):

soil_moisture    0.263723
temperature      0.251431
rainfall         0.212652
humidity         0.174370
ph               0.097824
dtype: float64


In [5]:
import matplotlib
matplotlib.use("Agg")  # headless-safe backend, no display required
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))
importances.sort_values().plot(kind="barh", ax=ax, color="#4C72B0")
ax.set_xlabel("Feature importance")
ax.set_title("Random Forest feature importance -- crop prediction")
fig.tight_layout()
fig.savefig("crop_feature_importance.png", dpi=100)
plt.close(fig)
print("Saved plot to crop_feature_importance.png")


Saved plot to crop_feature_importance.png


## 5. Try a new sensor reading


In [6]:
new_reading = pd.DataFrame([{
    "rainfall": 195, "temperature": 26, "soil_moisture": 78, "humidity": 82, "ph": 6.1,
}])

predicted_crop = model.predict(new_reading)[0]
probs = model.predict_proba(new_reading)[0]

print(f"Sensor reading: {new_reading.iloc[0].to_dict()}\n")
print("Per-crop probability:")
for crop, p in sorted(zip(model.classes_, probs), key=lambda x: -x[1]):
    print(f"  {crop:8s}: {p:.3f}")
print(f"\n-> Recommended crop: {predicted_crop}")


Sensor reading: {'rainfall': 195.0, 'temperature': 26.0, 'soil_moisture': 78.0, 'humidity': 82.0, 'ph': 6.1}

Per-crop probability:
  rice    : 1.000
  cotton  : 0.000
  maize   : 0.000
  millet  : 0.000
  wheat   : 0.000

-> Recommended crop: rice


## Takeaways

- Random Forest needs no feature scaling and handles the nonlinear, conjunctive relationships
  between sensor readings and crop suitability natively (e.g. "high rainfall AND high humidity
  favors rice") without hand-engineered interaction terms.
- `stratify=y` in the train/test split matters even with a fairly balanced synthetic dataset like
  this one -- it's a habit worth keeping regardless of how balanced you *think* your data is.
- The confusion matrix is more informative than accuracy alone: crops with genuinely similar growing
  conditions in this synthetic data (e.g. maize and cotton, both moderate-rainfall/moderate-humidity)
  are more likely to be confused with each other than with a very different crop like rice -- which
  is a sensible, low-severity kind of error compared to confusing a water-loving crop with a
  drought-tolerant one.
- Feature importance is a genuine agronomic sanity check as well as a model diagnostic -- see
  chapter 06 for the caveat about distinguishing "this feature doesn't matter" from "this dataset
  doesn't have enough variation in this feature for the model to learn its effect."
